# GRPO Training — Quartermaster Environment

**Group Relative Policy Optimization** with real environment rewards.

Each training example is one `(system + observation)` prompt. GRPO generates G completions online,
each is parsed into an action, the env is replayed to that step, stepped once, and the **real environment reward** is returned.

**Training tricks:**
- **DAPO Asymmetric Clipping** — prevents entropy collapse
- **Dr. GRPO Loss** — removes response-length bias
- **Truncation Masking** — avoids penalizing truncated completions

**Input:** `grpo_data.jsonl` (from `generate_grpo_data.ipynb`) + SFT model (from `sft_training.ipynb`)

**Output:** `./grpo_model/` + training dashboards

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install unsloth trl datasets matplotlib python-dotenv

## 1. Configuration

In [ ]:
import os
import json
import time
import logging

from dotenv import load_dotenv
load_dotenv()

# --- Config ---
SFT_MODEL_DIR = os.getenv("SFT_MODEL_DIR", "./sft_model_merged")
GRPO_DATA_FILE = os.getenv("GRPO_DATA_FILE", "grpo_data.jsonl")
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "./grpo_model")
NUM_EPOCHS = int(os.getenv("NUM_EPOCHS", "1"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "1"))
GRAD_ACCUM = int(os.getenv("GRAD_ACCUM", "4"))
MAX_SEQ_LENGTH = int(os.getenv("MAX_SEQ_LENGTH", "4096"))
MAX_COMPLETION = int(os.getenv("MAX_COMPLETION", "1024"))
NUM_GENERATIONS = int(os.getenv("NUM_GENERATIONS", "4"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "5e-6"))
LORA_RANK = int(os.getenv("LORA_RANK", "32"))

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("grpo_train")

# Reward tracking for plots
_reward_log = []
_component_log = []
_step_counter = [0]

print(f"SFT model: {SFT_MODEL_DIR}")
print(f"GRPO data: {GRPO_DATA_FILE}")
print(f"Output: {OUTPUT_DIR}")
print(f"Generations per prompt: {NUM_GENERATIONS}")
print(f"Max seq/completion: {MAX_SEQ_LENGTH}/{MAX_COMPLETION}")
print(f"LoRA rank: {LORA_RANK}")
print(f"Learning rate: {LEARNING_RATE}")

## 2. Import Environment & Inference Utilities

In [ ]:
from server.inventory_env import InventoryEnvironment
from models import InventoryAction
from inference import SYSTEM_PROMPT, format_observation, parse_action

## 3. Replay & Reward Functions

**How it works:**
1. `replay_to_step()` — Replays baseline actions to reconstruct env state at a given step
2. `step_reward()` — For each GRPO completion: parse action → replay env → step once → return real reward

This means every reward signal comes from the **actual environment**, not an approximation.

In [ ]:
def replay_to_step(task_name, prior_actions_json):
    """Replay baseline actions to reconstruct environment state at a given step."""
    prior_actions = json.loads(prior_actions_json)
    env = InventoryEnvironment(task_name)
    env.reset()
    for action_dict in prior_actions:
        action = InventoryAction(**action_dict)
        env.step(action)
    return env


def step_reward(completions, task_name, prior_actions, **kwargs):
    """Score each completion by parsing it, replaying env, stepping once.

    Args:
        completions: list of model-generated text (G per prompt)
        task_name: list of task names (dataset column, repeated for G completions)
        prior_actions: list of JSON-encoded action histories (dataset column)

    Returns:
        list of float rewards from the real environment
    """
    rewards = []
    batch_components = []
    for completion, tname, actions_json in zip(completions, task_name, prior_actions):
        try:
            env = replay_to_step(tname, actions_json)

            # Handle conversational format
            if isinstance(completion, list):
                text = completion[0]["content"] if completion else ""
            else:
                text = str(completion)

            action = parse_action(text)
            obs = env.step(action)
            rewards.append(obs.reward)
            batch_components.append(getattr(env, "reward_components", {}))
        except Exception as e:
            log.debug(f"Reward computation failed: {e}")
            rewards.append(-1.0)
            batch_components.append({})

    # Log batch stats
    _step_counter[0] += 1
    avg_r = sum(rewards) / len(rewards) if rewards else 0
    min_r = min(rewards) if rewards else 0
    max_r = max(rewards) if rewards else 0
    _reward_log.append({
        "step": _step_counter[0],
        "mean": avg_r, "min": min_r, "max": max_r,
        "rewards": rewards,
        "tasks": list(task_name),
    })

    # Log reward sub-components
    if batch_components and any(batch_components):
        comp_means = {}
        for key in ["R_directives", "R_planning", "R_revenue", "R_fulfillment", "R_waste",
                     "milestone_bonus", "directive_penalty", "hard_penalty"]:
            vals = [c.get(key, 0.0) for c in batch_components if c]
            comp_means[key] = sum(vals) / len(vals) if vals else 0.0
        comp_means["step"] = _step_counter[0]
        _component_log.append(comp_means)

    if _step_counter[0] % 5 == 0 or _step_counter[0] <= 3:
        log.info(
            f"[GRPO step {_step_counter[0]}] "
            f"reward: mean={avg_r:.3f} min={min_r:.3f} max={max_r:.3f} "
            f"(G={len(rewards)})"
        )

    return rewards

## 4. Load GRPO Dataset

In [ ]:
from datasets import Dataset

examples = []
task_counts = {}
with open(GRPO_DATA_FILE) as f:
    for line in f:
        row = json.loads(line.strip())
        examples.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": row["observation"]},
            ],
            "task_name": row["task_name"],
            "prior_actions": row["prior_actions"],
        })
        task_counts[row["task_name"]] = task_counts.get(row["task_name"], 0) + 1

dataset = Dataset.from_list(examples)

print(f"Loaded {len(examples)} prompts")
for task, count in task_counts.items():
    print(f"  {task}: {count} prompts")

## 5. Load SFT Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.9,
)

print(f"SFT model loaded from: {SFT_MODEL_DIR}")

## 6. Apply LoRA for GRPO

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 7. Configure GRPO Trainer

Three research-backed optimizations:

| Trick | What It Does |
|-------|--------------|
| **DAPO Asymmetric Clipping** (`epsilon_low=0.2, epsilon_high=0.28`) | Widens upper clipping bound to prevent entropy collapse — preserves exploration of novel strategies |
| **Dr. GRPO Loss** (`loss_type="dr_grpo"`) | Fixed denominator normalization removes response-length bias — longer wrong answers don't get softer gradients |
| **Truncation Masking** (`mask_truncated_completions=True`) | Truncated completions are masked out of loss, avoiding confusion between "bad reasoning" and "ran out of tokens" |

In [ ]:
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    bf16=True,
    logging_steps=1,
    save_strategy="epoch",
    max_completion_length=MAX_COMPLETION,
    num_generations=NUM_GENERATIONS,
    report_to="none",
    use_vllm=False,  # Unsloth's fast_inference handles vLLM internally
    # --- GRPO Training Tricks ---
    epsilon_low=0.2,
    epsilon_high=0.28,
    loss_type="dr_grpo",
    mask_truncated_completions=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=step_reward,
    args=training_args,
    train_dataset=dataset,
)

print("GRPO Trainer configured. Ready to train.")

## 8. Train

In [ ]:
train_start = time.time()
log.info("Starting GRPO training...")

trainer.train()

train_time = time.time() - train_start
log.info(f"GRPO training finished in {train_time:.1f}s ({train_time/60:.1f}min)")

## 9. Save Model

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## 10. Reward Dashboard

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _smooth(values, window):
    smoothed = []
    for i in range(len(values)):
        start = max(0, i - window)
        smoothed.append(sum(values[start:i+1]) / (i - start + 1))
    return smoothed

os.makedirs(OUTPUT_DIR, exist_ok=True)

if _reward_log:
    steps = [r["step"] for r in _reward_log]
    means = [r["mean"] for r in _reward_log]
    mins = [r["min"] for r in _reward_log]
    maxs = [r["max"] for r in _reward_log]
    window = max(5, len(means) // 15) if len(means) > 10 else 1

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("GRPO Training — QuarterMaster (Rewards)", fontsize=16, fontweight="bold")

    # Mean reward
    ax = axes[0, 0]
    ax.plot(steps, means, color="#3b82f6", linewidth=1, alpha=0.4, label="Raw mean")
    if len(means) > 10:
        ax.plot(steps, _smooth(means, window), color="#ef4444", linewidth=2.5, label=f"Smoothed (w={window})")
    ax.set_xlabel("Step")
    ax.set_ylabel("Mean Reward")
    ax.set_title("Mean Reward per Step")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Min/Max band
    ax = axes[0, 1]
    ax.fill_between(steps, mins, maxs, alpha=0.2, color="#8b5cf6", label="Min-Max range")
    ax.plot(steps, means, color="#3b82f6", linewidth=1.5, label="Mean")
    ax.set_xlabel("Step")
    ax.set_ylabel("Reward")
    ax.set_title("Reward Range")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Histogram
    ax = axes[1, 0]
    all_rewards = []
    for r in _reward_log:
        all_rewards.extend(r["rewards"])
    ax.hist(all_rewards, bins=50, color="#22c55e", alpha=0.7, edgecolor="#000")
    mean_all = sum(all_rewards) / len(all_rewards) if all_rewards else 0
    ax.axvline(x=mean_all, color="#ef4444", linestyle="--", linewidth=2, label=f"Mean={mean_all:.3f}")
    ax.set_xlabel("Reward")
    ax.set_ylabel("Count")
    ax.set_title("Reward Distribution (All Completions)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Cumulative mean
    ax = axes[1, 1]
    cumulative = []
    running = 0
    for i, m in enumerate(means):
        running += m
        cumulative.append(running / (i + 1))
    ax.plot(steps, cumulative, color="#f59e0b", linewidth=2)
    ax.set_xlabel("Step")
    ax.set_ylabel("Cumulative Mean Reward")
    ax.set_title("Cumulative Mean Reward")
    ax.grid(True, alpha=0.3)

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(os.path.join(OUTPUT_DIR, "grpo_reward_dashboard.png"), dpi=150)
    plt.show()
    print(f"Reward dashboard saved to {OUTPUT_DIR}/grpo_reward_dashboard.png")
else:
    print("No reward data to plot.")

## 11. Optimization Dashboard

In [ ]:
history = trainer.state.log_history if trainer else []
loss_steps = [h["step"] for h in history if "loss" in h]
losses = [h["loss"] for h in history if "loss" in h]
kl_steps = [h["step"] for h in history if "kl" in h]
kls = [h["kl"] for h in history if "kl" in h]
lr_steps = [h["step"] for h in history if "learning_rate" in h]
lrs = [h["learning_rate"] for h in history if "learning_rate" in h]
entropy_steps = [h["step"] for h in history if "entropy" in h]
entropies = [h["entropy"] for h in history if "entropy" in h]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("GRPO Training — QuarterMaster (Optimization)", fontsize=16, fontweight="bold")

# Loss
ax = axes[0, 0]
if loss_steps:
    ax.plot(loss_steps, losses, color="#3b82f6", linewidth=1, alpha=0.4, label="Raw loss")
    if len(losses) > 10:
        ax.plot(loss_steps, _smooth(losses, max(5, len(losses)//20)), color="#ef4444", linewidth=2.5, label="Smoothed")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No loss data", ha="center", va="center", transform=ax.transAxes, color="#888")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Policy Loss")
ax.grid(True, alpha=0.3)

# KL
ax = axes[0, 1]
if kl_steps:
    ax.plot(kl_steps, kls, color="#a855f7", linewidth=1, alpha=0.5)
    if len(kls) > 10:
        ax.plot(kl_steps, _smooth(kls, max(5, len(kls)//20)), color="#c084fc", linewidth=2.5, label="Smoothed")
        ax.legend()
else:
    ax.text(0.5, 0.5, "No KL data", ha="center", va="center", transform=ax.transAxes, color="#888")
ax.set_xlabel("Step")
ax.set_ylabel("KL Divergence")
ax.set_title("KL Divergence from Reference")
ax.grid(True, alpha=0.3)

# LR
ax = axes[1, 0]
if lr_steps:
    ax.plot(lr_steps, lrs, color="#06b6d4", linewidth=2)
    ax.ticklabel_format(style="scientific", axis="y", scilimits=(0, 0))
else:
    ax.text(0.5, 0.5, "No LR data", ha="center", va="center", transform=ax.transAxes, color="#888")
ax.set_xlabel("Step")
ax.set_ylabel("Learning Rate")
ax.set_title("Learning Rate Schedule")
ax.grid(True, alpha=0.3)

# Entropy
ax = axes[1, 1]
if entropy_steps:
    ax.plot(entropy_steps, entropies, color="#f59e0b", linewidth=1, alpha=0.5)
    if len(entropies) > 10:
        ax.plot(entropy_steps, _smooth(entropies, max(5, len(entropies)//20)), color="#ef4444", linewidth=2.5, label="Smoothed")
        ax.legend()
else:
    ax.text(0.5, 0.5, "No entropy data", ha="center", va="center", transform=ax.transAxes, color="#888")
ax.set_xlabel("Step")
ax.set_ylabel("Entropy")
ax.set_title("Policy Entropy")
ax.grid(True, alpha=0.3)

fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(os.path.join(OUTPUT_DIR, "grpo_optimization_dashboard.png"), dpi=150)
plt.show()
print(f"Optimization dashboard saved to {OUTPUT_DIR}/grpo_optimization_dashboard.png")

## 12. Per-Task Rewards

In [ ]:
task_rewards_by_step = {}
for entry in _reward_log:
    for task, reward in zip(entry.get("tasks", []), entry["rewards"]):
        task_rewards_by_step.setdefault(task, []).append(reward)

if task_rewards_by_step:
    task_colors = {"easy": "#22c55e", "medium": "#f59e0b", "hard": "#ef4444"}
    n_tasks = len(task_rewards_by_step)
    fig, axes = plt.subplots(1, max(n_tasks, 1) + 1, figsize=(6 * (n_tasks + 1), 5))
    fig.suptitle("GRPO Training — Per-Task Reward Analysis", fontsize=14, fontweight="bold")
    if n_tasks + 1 == 1:
        axes = [axes]

    for i, (task, rewards) in enumerate(sorted(task_rewards_by_step.items())):
        ax = axes[i]
        color = task_colors.get(task, "#888")
        ax.hist(rewards, bins=30, color=color, alpha=0.7, edgecolor="#000")
        mean_r = sum(rewards) / len(rewards)
        ax.axvline(x=mean_r, color="#000", linestyle="--", linewidth=2, label=f"Mean={mean_r:.3f}")
        ax.set_xlabel("Reward")
        ax.set_ylabel("Count")
        ax.set_title(f"{task.title()} (n={len(rewards)})")
        ax.legend()
        ax.grid(True, alpha=0.3)

    # Summary bar
    ax = axes[-1]
    labels, bar_means, bar_colors = [], [], []
    for task in ["easy", "medium", "hard"]:
        if task in task_rewards_by_step:
            r = task_rewards_by_step[task]
            labels.append(task)
            bar_means.append(sum(r) / len(r))
            bar_colors.append(task_colors.get(task, "#888"))
    ax.bar(labels, bar_means, color=bar_colors, width=0.5)
    ax.set_ylabel("Mean Reward")
    ax.set_title("Mean Reward Comparison")
    ax.grid(True, alpha=0.3, axis="y")

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(os.path.join(OUTPUT_DIR, "grpo_per_task_rewards.png"), dpi=150)
    plt.show()
    print(f"Per-task rewards saved to {OUTPUT_DIR}/grpo_per_task_rewards.png")
else:
    print("No per-task data to plot.")

## 12b. Per-Task Reward Curves Over Training

Shows how reward evolves over time for each task (smoothed + cumulative mean).

In [ ]:
if task_rewards_by_step:
    task_colors = {"easy": "#22c55e", "medium": "#f59e0b", "hard": "#ef4444"}
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("GRPO Training — Per-Task Reward Curves", fontsize=14, fontweight="bold")

    # Left: Rolling mean reward per task over time
    ax = axes[0]
    for task in ["easy", "medium", "hard"]:
        if task in task_rewards_by_step:
            rewards = task_rewards_by_step[task]
            color = task_colors.get(task, "#888")
            # Plot raw as faint dots
            ax.scatter(range(len(rewards)), rewards, color=color, alpha=0.1, s=4)
            # Smoothed curve
            if len(rewards) > 10:
                w = max(5, len(rewards) // 15)
                ax.plot(range(len(rewards)), _smooth(rewards, w),
                        color=color, linewidth=2.5, label=f"{task} (smoothed)")
            else:
                ax.plot(range(len(rewards)), rewards,
                        color=color, linewidth=1.5, label=task)
    ax.set_xlabel("Sample Index (within task)")
    ax.set_ylabel("Reward")
    ax.set_title("Reward Over Time (per task)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Right: Cumulative mean per task
    ax = axes[1]
    for task in ["easy", "medium", "hard"]:
        if task in task_rewards_by_step:
            rewards = task_rewards_by_step[task]
            color = task_colors.get(task, "#888")
            cum = []
            running = 0
            for i, r in enumerate(rewards):
                running += r
                cum.append(running / (i + 1))
            ax.plot(range(len(cum)), cum, color=color, linewidth=2, label=task)
    ax.set_xlabel("Sample Index (within task)")
    ax.set_ylabel("Cumulative Mean Reward")
    ax.set_title("Cumulative Mean Reward (per task)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(os.path.join(OUTPUT_DIR, "grpo_per_task_curves.png"), dpi=150)
    plt.show()
    print(f"Per-task curves saved to {OUTPUT_DIR}/grpo_per_task_curves.png")
else:
    print("No per-task data to plot.")

## 13. Reward Decomposition

Shows how each reward sub-component (directives, planning, revenue, fulfillment, waste) evolves during training.

In [ ]:
if _component_log and len(_component_log) > 5:
    dense_keys = ["R_directives", "R_planning", "R_revenue", "R_fulfillment", "R_waste"]
    sparse_keys = ["milestone_bonus", "directive_penalty", "hard_penalty"]
    comp_steps = [c["step"] for c in _component_log]
    colors_dense = {"R_directives": "#ef4444", "R_planning": "#8b5cf6",
                    "R_revenue": "#22c55e", "R_fulfillment": "#3b82f6", "R_waste": "#f59e0b"}
    weights = {"R_directives": 0.40, "R_planning": 0.20, "R_revenue": 0.15,
               "R_fulfillment": 0.15, "R_waste": 0.10}

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle("GRPO Training — Reward Decomposition", fontsize=14, fontweight="bold")

    # Dense components over time
    ax = axes[0]
    for key in dense_keys:
        vals = [c.get(key, 0.0) for c in _component_log]
        w = max(5, len(vals) // 15)
        ax.plot(comp_steps, _smooth(vals, w), color=colors_dense[key], linewidth=2,
                label=f"{key} ({weights[key]:.0%})")
    ax.set_xlabel("Step")
    ax.set_ylabel("Component Value")
    ax.set_title("Dense Reward Components")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color="#000", linewidth=0.5, alpha=0.5)

    # Weighted contributions
    ax = axes[1]
    for key in dense_keys:
        vals = [c.get(key, 0.0) * weights[key] for c in _component_log]
        w = max(5, len(vals) // 15)
        ax.plot(comp_steps, _smooth(vals, w), color=colors_dense[key], linewidth=2, label=key)
    ax.set_xlabel("Step")
    ax.set_ylabel("Weighted Contribution")
    ax.set_title("Weighted Dense Contributions")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color="#000", linewidth=0.5, alpha=0.5)

    # Mean bar chart
    ax = axes[2]
    all_keys = dense_keys + sparse_keys
    sparse_colors = {"milestone_bonus": "#22c55e", "directive_penalty": "#ef4444", "hard_penalty": "#f97316"}
    bar_means = []
    bar_colors = []
    for key in all_keys:
        vals = [c.get(key, 0.0) for c in _component_log]
        bar_means.append(sum(vals) / len(vals))
        bar_colors.append(colors_dense.get(key, sparse_colors.get(key, "#888")))
    short_labels = [k.replace("R_", "").replace("_", "\n") for k in all_keys]
    bars = ax.bar(short_labels, bar_means, color=bar_colors, width=0.6)
    for bar, val in zip(bars, bar_means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{val:.3f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=7, fontweight="bold")
    ax.set_ylabel("Mean Value")
    ax.set_title("Average Component Values")
    ax.grid(True, alpha=0.3, axis="y")
    ax.axhline(y=0, color="#000", linewidth=0.5)
    ax.tick_params(axis='x', labelsize=7)

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(os.path.join(OUTPUT_DIR, "grpo_reward_decomposition.png"), dpi=150)
    plt.show()
    print(f"Reward decomposition saved to {OUTPUT_DIR}/grpo_reward_decomposition.png")
else:
    print("Not enough component data to plot (need > 5 steps).")

## 14. Improvement Summary

Compare first half vs second half of training to quantify improvement.

In [ ]:
if _reward_log and len([r["mean"] for r in _reward_log]) >= 10:
    means = [r["mean"] for r in _reward_log]
    mid = len(means) // 2
    first_half = means[:mid]
    second_half = means[mid:]
    fh_mean = sum(first_half) / len(first_half)
    sh_mean = sum(second_half) / len(second_half)
    pct_change = ((sh_mean - fh_mean) / abs(fh_mean) * 100) if fh_mean != 0 else 0

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    bars = ax.bar(["First Half", "Second Half"], [fh_mean, sh_mean],
                  color=["#ef4444", "#22c55e"], width=0.5)
    ax.set_ylabel("Mean Reward")
    ax.set_title(f"Reward Improvement: {pct_change:+.1f}%")
    ax.grid(True, alpha=0.3, axis="y")
    for bar, val in zip(bars, [fh_mean, sh_mean]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "grpo_improvement.png"), dpi=150)
    plt.show()
    print(f"Improvement chart saved to {OUTPUT_DIR}/grpo_improvement.png")
else:
    print("Not enough data for improvement comparison (need >= 10 steps).")

# Final summary
if _reward_log:
    all_means = [r["mean"] for r in _reward_log]
    print("\n=== GRPO TRAINING SUMMARY ===")
    print(f"  Total steps: {len(_reward_log)}")
    print(f"  Overall mean reward: {sum(all_means)/len(all_means):.3f}")
    print(f"  First 10% mean: {sum(all_means[:max(1,len(all_means)//10)])/max(1,len(all_means)//10):.3f}")
    print(f"  Last 10% mean: {sum(all_means[-max(1,len(all_means)//10):])/max(1,len(all_means)//10):.3f}")
    print(f"  Best step reward: {max(all_means):.3f}")
    print(f"  Training time: {train_time/60:.1f} minutes")

## 15. Save Training Logs

In [ ]:
# Save reward log
with open(os.path.join(OUTPUT_DIR, "grpo_training_log.json"), "w") as f:
    json.dump(_reward_log, f, indent=2)
print(f"Reward log saved to {OUTPUT_DIR}/grpo_training_log.json")

# Save component log
if _component_log:
    with open(os.path.join(OUTPUT_DIR, "grpo_component_log.json"), "w") as f:
        json.dump(_component_log, f, indent=2)
    print(f"Component log saved to {OUTPUT_DIR}/grpo_component_log.json")

# Save trainer log history
if history:
    with open(os.path.join(OUTPUT_DIR, "grpo_trainer_log.json"), "w") as f:
        json.dump(history, f, indent=2)
    print(f"Trainer log saved to {OUTPUT_DIR}/grpo_trainer_log.json")

print(f"\nGRPO training complete. Model at: {OUTPUT_DIR}")